# Testes do motor de dosimetria

Cobre o motor de dosimetria inteiro (`dosimetria/`): tipos de valor, as três fases e a dosimetria completa. Rode as células em ordem ("Run All"); qualquer `assert` que falhar interrompe a execução naquele ponto e mostra o traceback.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "dosimetria").is_dir():
        sys.path.insert(0, str(candidate))
        break

from dosimetria import Fracao, Pena, Faixa

## Fracao

In [2]:
assert Fracao(1, 3).aplicar(360) == 120

# 1/3 de 365 = 121.66... -> desprezar a fração de dia (art. 11 do CP)
assert Fracao(1, 3).aplicar(365) == 121

assert Fracao(1, 2).aplicar(0) == 0

for numerador, denominador in [(1, 0), (1, -3)]:
    try:
        Fracao(numerador, denominador)
        raise AssertionError(f"deveria ter rejeitado denominador={denominador}")
    except ValueError:
        pass

try:
    Fracao(-1, 3)
    raise AssertionError("deveria ter rejeitado numerador negativo")
except ValueError:
    pass

assert str(Fracao(1, 6)) == "1/6"

print("Fracao: OK")

Fracao: OK


## Pena

In [3]:
# convenção do projeto: 1 ano = 365 dias, 1 mês = 30 dias
assert Pena.de_anos_meses_dias(anos=1).dias == 365
assert Pena.de_anos_meses_dias(meses=1).dias == 30
assert Pena.de_anos_meses_dias(anos=1, meses=2, dias=3).dias == 365 + 60 + 3

try:
    Pena(-1)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Pena(360).mais(Fracao(1, 3)).dias == 480
assert Pena(360).menos(Fracao(1, 6)).dias == 300

assert Pena(100).mais_dias(50).dias == 150
assert Pena(100).menos_dias(50).dias == 50

try:
    Pena(10).menos_dias(20)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Pena.de_anos_meses_dias(anos=1, meses=4, dias=4).como_anos_meses_dias() == (1, 4, 4)

assert Pena(100) < Pena(200)
assert Pena(200) > Pena(100)
assert Pena(100) <= Pena(100)
assert Pena(100) == Pena(100)

assert str(Pena.de_anos_meses_dias(anos=1, meses=4, dias=4)) == "1 ano, 4 meses, 4 dias"
assert str(Pena(0)) == "0 dias"

# 1 ano = 365 dias, mas 12 meses = 360: os meses param em 11 e o excedente fica nos dias
assert Pena(726).como_anos_meses_dias() == (1, 11, 31)
assert str(Pena(726)) == "1 ano, 11 meses, 31 dias"
assert Pena(364).como_anos_meses_dias() == (0, 11, 34)
assert Pena(359).como_anos_meses_dias() == (0, 11, 29)
assert Pena(360).como_anos_meses_dias() == (0, 11, 30)
assert Pena(365).como_anos_meses_dias() == (1, 0, 0)
# a decomposição continua exata: volta para o mesmo número de dias
for total in range(0, 3 * 365):
    anos, meses, dias = Pena(total).como_anos_meses_dias()
    assert meses <= 11
    assert Pena.de_anos_meses_dias(anos, meses, dias).dias == total

print("Pena: OK")

Pena: OK


## Faixa

In [4]:
faixa_furto_simples = Faixa(
    minimo=Pena.de_anos_meses_dias(anos=1),
    maximo=Pena.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

assert faixa_furto_simples.contem(Pena.de_anos_meses_dias(anos=2))
assert faixa_furto_simples.contem(faixa_furto_simples.minimo)
assert faixa_furto_simples.contem(faixa_furto_simples.maximo)
assert not faixa_furto_simples.contem(Pena.de_anos_meses_dias(anos=5))

assert faixa_furto_simples.limitar(Pena.de_anos_meses_dias(meses=6)) == faixa_furto_simples.minimo
assert faixa_furto_simples.limitar(Pena.de_anos_meses_dias(anos=10)) == faixa_furto_simples.maximo

pena_dentro = Pena.de_anos_meses_dias(anos=2)
assert faixa_furto_simples.limitar(pena_dentro) == pena_dentro

try:
    Faixa(minimo=Pena(100), maximo=Pena(50), origem="teste")
    raise AssertionError("deveria ter rejeitado mínimo > máximo")
except ValueError:
    pass

print("Faixa: OK")

Faixa: OK


## Quantum (estratégias de incremento por circunstância)

In [5]:
from dosimetria import EstrategiaQuantum, FracaoDoIntervalo, FracaoDoMinimo

faixa_furto = Faixa(
    minimo=Pena.de_anos_meses_dias(anos=1),
    maximo=Pena.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

# intervalo = 3 anos = 1095 dias; 1/8 do intervalo
assert FracaoDoIntervalo().incremento_por_circunstancia(faixa_furto).dias == Fracao(1, 8).aplicar(1095)

# 1/6 do mínimo (365 dias)
assert FracaoDoMinimo().incremento_por_circunstancia(faixa_furto).dias == Fracao(1, 6).aplicar(365)

assert isinstance(FracaoDoIntervalo(), EstrategiaQuantum)
assert "1/8" in FracaoDoIntervalo().nome

print("Quantum: OK")

Quantum: OK


## Fase 1 — pena-base (art. 59 do CP)

In [6]:
from dosimetria import CircunstanciaJudicial, Valoracao, calcular_pena_base

TODAS_NEUTRAS = {c: Valoracao.NEUTRA for c in CircunstanciaJudicial}


def com(desfavoraveis):
    valores = dict(TODAS_NEUTRAS)
    for circunstancia in desfavoraveis:
        valores[circunstancia] = Valoracao.DESFAVORAVEL
    return valores


# nenhuma circunstância desfavorável -> pena-base no mínimo da faixa
resultado = calcular_pena_base(faixa_furto, TODAS_NEUTRAS, FracaoDoIntervalo())
assert resultado.pena_base == faixa_furto.minimo
assert resultado.passo.valor_antes == faixa_furto.minimo
assert resultado.passo.valor_depois == faixa_furto.minimo
assert resultado.passo.dispositivo == "CP.art155"

# uma circunstância desfavorável, estratégia 1/8 do intervalo
circunstancias_1 = com([CircunstanciaJudicial.CULPABILIDADE])
resultado_1 = calcular_pena_base(faixa_furto, circunstancias_1, FracaoDoIntervalo())
incremento_esperado = Fracao(1, 8).aplicar(1095)
assert resultado_1.pena_base.dias == faixa_furto.minimo.dias + incremento_esperado

# todas as 8 desfavoráveis nunca ultrapassa o máximo da faixa (mesmo com truncamento por fração)
todas_desfavoraveis = {c: Valoracao.DESFAVORAVEL for c in CircunstanciaJudicial}
resultado_max = calcular_pena_base(faixa_furto, todas_desfavoraveis, FracaoDoIntervalo())
assert resultado_max.pena_base <= faixa_furto.maximo

# com um intervalo múltiplo de 8, 8 circunstâncias desfavoráveis batem exatamente no máximo
faixa_multipla_de_8 = Faixa(minimo=Pena(0), maximo=Pena(800), origem="teste")
resultado_max_exato = calcular_pena_base(faixa_multipla_de_8, todas_desfavoraveis, FracaoDoIntervalo())
assert resultado_max_exato.pena_base == faixa_multipla_de_8.maximo

# circunstância faltando ou desconhecida é rejeitada
try:
    calcular_pena_base(faixa_furto, {CircunstanciaJudicial.CULPABILIDADE: Valoracao.NEUTRA}, FracaoDoIntervalo())
    raise AssertionError("deveria exigir as 8 circunstâncias")
except ValueError:
    pass

print("Fase 1 (pena-base): OK")

Fase 1 (pena-base): OK


## Fase 2 — pena intermediária (agravantes e atenuantes, arts. 61 a 67)

In [7]:
from dosimetria import CircunstanciaLegal, Direcao, calcular_pena_intermediaria

estrategia = FracaoDoIntervalo()
incremento = estrategia.incremento_por_circunstancia(faixa_furto).dias  # 136
pena_base_1_desfavoravel = resultado_1.pena_base  # 365 + 136 = 501


def agravante(codigo="reincidencia", preponderante=False):
    return CircunstanciaLegal(codigo, "CP.art61.I", Direcao.AGRAVANTE, preponderante)


def atenuante(codigo="confissao_espontanea", preponderante=False):
    return CircunstanciaLegal(codigo, "CP.art65.III.d", Direcao.ATENUANTE, preponderante)


# sem agravantes nem atenuantes -> pena intermediária = pena-base
sem_nada = calcular_pena_intermediaria(faixa_furto, pena_base_1_desfavoravel, [], estrategia)
assert sem_nada.pena_intermediaria == pena_base_1_desfavoravel

# uma agravante -> soma um incremento
so_agravante = calcular_pena_intermediaria(faixa_furto, pena_base_1_desfavoravel, [agravante()], estrategia)
assert so_agravante.pena_intermediaria.dias == pena_base_1_desfavoravel.dias + incremento

# uma atenuante que empurraria abaixo do mínimo -> Súmula 231, trava no mínimo
perto_do_minimo = calcular_pena_intermediaria(faixa_furto, faixa_furto.minimo, [atenuante()], estrategia)
assert perto_do_minimo.pena_intermediaria == faixa_furto.minimo

# agravante preponderante + atenuante não preponderante -> só a agravante conta (art. 67)
concurso_agravante_prepondera = calcular_pena_intermediaria(
    faixa_furto,
    pena_base_1_desfavoravel,
    [agravante(preponderante=True), atenuante(preponderante=False)],
    estrategia,
)
assert concurso_agravante_prepondera.pena_intermediaria.dias == pena_base_1_desfavoravel.dias + incremento
assert "preponderam as agravantes" in concurso_agravante_prepondera.passo.motivo

# atenuante preponderante + agravante não preponderante -> só a atenuante conta (art. 67)
concurso_atenuante_prepondera = calcular_pena_intermediaria(
    faixa_furto,
    pena_base_1_desfavoravel,
    [agravante(preponderante=False), atenuante(preponderante=True)],
    estrategia,
)
assert concurso_atenuante_prepondera.pena_intermediaria.dias == pena_base_1_desfavoravel.dias - incremento

# nenhuma das duas é preponderante -> compensação líquida (uma cancela a outra)
compensacao = calcular_pena_intermediaria(
    faixa_furto, pena_base_1_desfavoravel, [agravante(), atenuante()], estrategia
)
assert compensacao.pena_intermediaria == pena_base_1_desfavoravel

print("Fase 2 (pena intermediária): OK")

Fase 2 (pena intermediária): OK


## Fase 3 — pena definitiva (causas de aumento e diminuição, art. 68)

In [8]:
from dosimetria import (
    CausaModificadora,
    Composicao,
    DirecaoCausa,
    OrigemCausa,
    calcular_pena_definitiva,
)

AUMENTO, DIMINUICAO = DirecaoCausa.AUMENTO, DirecaoCausa.DIMINUICAO
GERAL, ESPECIAL = OrigemCausa.PARTE_GERAL, OrigemCausa.PARTE_ESPECIAL

repouso_noturno = CausaModificadora("repouso_noturno", "CP.art155.§1", AUMENTO, ESPECIAL, Fracao(1, 3))


def tentativa(escolhida=None, justificativa=None):
    return CausaModificadora(
        "tentativa", "CP.art14.parágrafo_único", DIMINUICAO, GERAL,
        fracao_min=Fracao(1, 3), fracao_max=Fracao(2, 3),
        fracao_escolhida=escolhida, justificativa=justificativa,
    )


# sem causas -> pena definitiva = pena intermediária, com um passo explicando
sem_causas = calcular_pena_definitiva(Pena(365), [], Composicao.CASCATA)
assert sem_causas.aplicando_todas.pena_definitiva == Pena(365)
assert len(sem_causas.aplicando_todas.passos) == 1
assert sem_causas.limitada_art68 is None

# furto noturno: 365 + 1/3 = 486,67 -> 486 (art. 11)
noturno = calcular_pena_definitiva(Pena(365), [repouso_noturno], Composicao.CASCATA)
assert noturno.aplicando_todas.pena_definitiva == Pena(486)
assert noturno.aplicando_todas.passos[0].dispositivo == "CP.art155.§1"

# 3ª fase pode ultrapassar o máximo da faixa...
acima = calcular_pena_definitiva(faixa_furto.maximo, [repouso_noturno], Composicao.CASCATA)
assert acima.aplicando_todas.pena_definitiva == Pena(1946)
assert not faixa_furto.contem(acima.aplicando_todas.pena_definitiva)

# ...e ficar abaixo do mínimo (tentativa, fração mínima por padrão: 365 - 1/3 = 243,33 -> 243)
abaixo = calcular_pena_definitiva(faixa_furto.minimo, [tentativa()], Composicao.CASCATA)
assert abaixo.aplicando_todas.pena_definitiva == Pena(243)
assert not faixa_furto.contem(abaixo.aplicando_todas.pena_definitiva)

# fração acima da mínima sem justificativa é rejeitada; fora do intervalo legal também
for escolhida, justificativa in [(Fracao(2, 3), None), (Fracao(2, 3), "   "), (Fracao(3, 4), "qualquer")]:
    try:
        tentativa(escolhida, justificativa)
        raise AssertionError(f"deveria ter rejeitado {escolhida} com justificativa={justificativa!r}")
    except ValueError:
        pass

# com justificativa, a fração escolhida vale e a justificativa vai para o relatório
justificada = calcular_pena_definitiva(
    Pena(365), [tentativa(Fracao(2, 3), "iter criminis mal iniciado")], Composicao.CASCATA
)
assert justificada.aplicando_todas.pena_definitiva == Pena(121)  # 365/3 = 121,67
assert "iter criminis mal iniciado" in justificada.aplicando_todas.passos[0].motivo

# cascata x sobre a pena intermediária: +1/3 e -1/3 sobre 360
causas_mistas = [repouso_noturno, tentativa()]
assert calcular_pena_definitiva(Pena(360), causas_mistas, Composicao.CASCATA).aplicando_todas.pena_definitiva == Pena(320)
assert calcular_pena_definitiva(Pena(360), causas_mistas, Composicao.SOBRE_PENA_INTERMEDIARIA).aplicando_todas.pena_definitiva == Pena(360)

# arredonda uma vez só, no fim da fase: 365 * 7/6 * 7/6 = 496,8 -> 496
# (truncando a cada passo daria 425 -> 495); por isso a ordem das causas não importa
concurso_formal = CausaModificadora("concurso_formal", "CP.art70", AUMENTO, GERAL, Fracao(1, 6), Fracao(1, 2))
continuidade = CausaModificadora("crime_continuado", "CP.art71", AUMENTO, GERAL, Fracao(1, 6), Fracao(2, 3))
duas = calcular_pena_definitiva(Pena(365), [concurso_formal, continuidade], Composicao.CASCATA)
invertidas = calcular_pena_definitiva(Pena(365), [continuidade, concurso_formal], Composicao.CASCATA)
assert duas.aplicando_todas.pena_definitiva == Pena(496)
assert invertidas.aplicando_todas.pena_definitiva == Pena(496)
assert duas.limitada_art68 is None  # causas da Parte Geral não entram no art. 68, parágrafo único

# os passos encadeiam: o "depois" de um é o "antes" do seguinte
passos = duas.aplicando_todas.passos
assert passos[0].valor_antes == Pena(365)
assert passos[-1].valor_depois == duas.aplicando_todas.pena_definitiva
assert all(a.valor_depois == b.valor_antes for a, b in zip(passos, passos[1:]))

# roubo com concurso de pessoas (1/3 a 1/2) + arma de fogo (2/3), ambas da Parte Especial,
# e tentativa (Parte Geral): o motor mostra as duas opções do art. 68, parágrafo único
concurso_de_pessoas = CausaModificadora(
    "concurso_de_pessoas", "CP.art157.§2.II", AUMENTO, ESPECIAL, Fracao(1, 3), Fracao(1, 2)
)
arma_de_fogo = CausaModificadora("arma_de_fogo", "CP.art157.§2-A.I", AUMENTO, ESPECIAL, Fracao(2, 3))
roubo = calcular_pena_definitiva(
    Pena(1460), [concurso_de_pessoas, arma_de_fogo, tentativa()], Composicao.CASCATA
)
assert roubo.aplicando_todas.pena_definitiva == Pena(2162)  # 1460 * 4/3 * 5/3 * 2/3 = 2162,96
assert roubo.limitada_art68 is not None
assert roubo.limitada_art68.pena_definitiva == Pena(1622)  # 1460 * 5/3 * 2/3 = 1622,2 (prevalece a arma)
assert any(
    p.regra.startswith("art. 68, parágrafo único") and "concurso_de_pessoas" in p.motivo
    for p in roubo.limitada_art68.passos
)
assert not any("concurso_de_pessoas" in p.motivo for p in roubo.limitada_art68.passos if p.regra == "art. 68 do CP")

# concurso de diminuições da Parte Especial: prevalece a que mais diminui
dim_menor = CausaModificadora("dim_menor", "teste.a", DIMINUICAO, ESPECIAL, Fracao(1, 6))
dim_maior = CausaModificadora("dim_maior", "teste.b", DIMINUICAO, ESPECIAL, Fracao(1, 3))
diminuicoes = calcular_pena_definitiva(Pena(360), [dim_menor, dim_maior], Composicao.CASCATA)
assert diminuicoes.aplicando_todas.pena_definitiva == Pena(200)  # 360 * 5/6 * 2/3
assert diminuicoes.limitada_art68.pena_definitiva == Pena(240)  # 360 * 2/3

# diminuições que somam mais de 100% sobre a pena intermediária são rejeitadas (na cascata, não)
dois_tercos = CausaModificadora("dim_2_3", "teste.c", DIMINUICAO, GERAL, Fracao(2, 3))
try:
    calcular_pena_definitiva(Pena(360), [dois_tercos, dois_tercos], Composicao.SOBRE_PENA_INTERMEDIARIA)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass
assert calcular_pena_definitiva(Pena(360), [dois_tercos, dois_tercos], Composicao.CASCATA).aplicando_todas.pena_definitiva == Pena(40)

print("Fase 3 (pena definitiva): OK")

Fase 3 (pena definitiva): OK


## Dosimetria completa (as três fases encadeadas + alertas)

In [9]:
from dosimetria import ResultadoDosimetria, calcular_dosimetria_completa

faixa_roubo = Faixa(
    minimo=Pena.de_anos_meses_dias(anos=4),
    maximo=Pena.de_anos_meses_dias(anos=10),
    origem="CP.art157",
)

# roubo: culpabilidade desfavorável (1/8 do intervalo de 2190 dias = 273), confissão,
# concurso de pessoas + arma de fogo (concurso de causas da Parte Especial)
roubo_completo = calcular_dosimetria_completa(
    faixa_roubo,
    com([CircunstanciaJudicial.CULPABILIDADE]),
    [atenuante()],
    [concurso_de_pessoas, arma_de_fogo],
    FracaoDoIntervalo(),
    Composicao.CASCATA,
)
assert isinstance(roubo_completo, ResultadoDosimetria)
assert roubo_completo.faixa_aplicada == faixa_roubo
assert roubo_completo.pena_base == Pena(1460 + 273)
assert roubo_completo.pena_intermediaria == Pena(1460)
assert roubo_completo.pena_definitiva == Pena(3244)  # 1460 * 4/3 * 5/3 = 3244,4
assert roubo_completo.alternativa_art68.pena_definitiva == Pena(2433)  # 1460 * 5/3 = 2433,3
assert roubo_completo.criterio_quantum == FracaoDoIntervalo().nome
assert roubo_completo.composicao is Composicao.CASCATA

# passo a passo completo e encadeado: mínimo da faixa -> pena-base -> intermediária -> definitiva
passos = roubo_completo.passos
assert [p.fase for p in passos] == [
    "1ª fase (pena-base)",
    "2ª fase (pena intermediária)",
    "3ª fase (pena definitiva)",
    "3ª fase (pena definitiva)",
]
assert passos[0].valor_antes == faixa_roubo.minimo
assert passos[-1].valor_depois == roubo_completo.pena_definitiva
assert all(a.valor_depois == b.valor_antes for a, b in zip(passos, passos[1:]))
assert any("art. 68, parágrafo único" in alerta for alerta in roubo_completo.alertas)

# furto sem nada desfavorável + confissão: Súmula 231 trava no mínimo e vira alerta
furto_231 = calcular_dosimetria_completa(
    faixa_furto, TODAS_NEUTRAS, [atenuante()], [], FracaoDoIntervalo(), Composicao.CASCATA
)
assert furto_231.pena_definitiva == faixa_furto.minimo
assert furto_231.alternativa_art68 is None
assert len(furto_231.alertas) == 1 and "Súmula 231" in furto_231.alertas[0]

# furto com as 8 desfavoráveis + reincidência + repouso noturno:
# a 2ª fase trava no máximo, e a 3ª fase passa dele (as duas coisas viram alerta)
furto_maximo = calcular_dosimetria_completa(
    faixa_furto, todas_desfavoraveis, [agravante()], [repouso_noturno], FracaoDoIntervalo(), Composicao.CASCATA
)
assert furto_maximo.pena_intermediaria == faixa_furto.maximo
assert furto_maximo.pena_definitiva == Pena(1946)
assert any("máximo da faixa" in alerta and "agravante" in alerta for alerta in furto_maximo.alertas)
assert any("acima do máximo" in alerta for alerta in furto_maximo.alertas)

# quantum de 1/4 do intervalo com as 8 desfavoráveis estoura a faixa na 1ª fase -> alerta
furto_quantum_alto = calcular_dosimetria_completa(
    faixa_furto, todas_desfavoraveis, [], [], FracaoDoIntervalo(Fracao(1, 4)), Composicao.CASCATA
)
assert furto_quantum_alto.pena_base == faixa_furto.maximo
assert any(alerta.startswith("pena-base calculada") for alerta in furto_quantum_alto.alertas)

print("Dosimetria completa: OK")

Dosimetria completa: OK


## Fundamentação (texto da dosimetria para leitura humana)

In [10]:
from dosimetria import gerar_fundamentacao

# roubo com duas majorantes da Parte Especial: todas as seções aparecem
texto = gerar_fundamentacao(roubo_completo)
print(texto)

assert texto.startswith("DOSIMETRIA DA PENA")
assert f"Faixa aplicada (CP.art157): de {faixa_roubo.minimo} a {faixa_roubo.maximo}." in texto
assert roubo_completo.criterio_quantum in texto
assert f"Pena-base: {roubo_completo.pena_base}." in texto
assert f"Pena intermediária: {roubo_completo.pena_intermediaria}." in texto
assert f"Pena definitiva: {roubo_completo.pena_definitiva}." in texto
assert f"Pena definitiva nesta opção: {roubo_completo.alternativa_art68.pena_definitiva}." in texto
for alerta in roubo_completo.alertas:
    assert f"- {alerta}" in texto

# as fases aparecem na ordem, e cada passo do resultado vira uma linha do texto
posicoes = [texto.index(titulo) for titulo in ["1ª FASE", "2ª FASE", "3ª FASE", "OPÇÃO DO ART. 68", "ALERTAS"]]
assert posicoes == sorted(posicoes)
for passo in roubo_completo.passos:
    assert f"- {passo.motivo} [{passo.regra}; {passo.dispositivo}]: {passo.valor_antes} -> {passo.valor_depois}" in texto

# os motivos das fases 1 e 2 nomeiam as circunstâncias
assert "(culpabilidade)" in roubo_completo.passos[0].motivo
assert "(confissao_espontanea)" in roubo_completo.passos[1].motivo

# sem alternativa nem alertas, essas seções não aparecem
texto_simples = gerar_fundamentacao(
    calcular_dosimetria_completa(faixa_furto, TODAS_NEUTRAS, [], [], FracaoDoIntervalo(), Composicao.CASCATA)
)
assert "OPÇÃO DO ART. 68" not in texto_simples
assert "ALERTAS" not in texto_simples
assert "nenhuma circunstância desfavorável" in texto_simples

print()
print("Fundamentação: OK")

DOSIMETRIA DA PENA

Faixa aplicada (CP.art157): de 4 anos a 10 anos.
Critério de quantum nas fases 1 e 2: fração do intervalo (1/8).
Composição da 3ª fase: causas compostas em cascata (cada fração sobre a pena já modificada).

1ª FASE (PENA-BASE)
- 1 circunstância(s) desfavorável(is) do art. 59 (culpabilidade), fração do intervalo (1/8) cada [art. 59 do CP; CP.art157]: 4 anos -> 4 anos, 9 meses, 3 dias
Pena-base: 4 anos, 9 meses, 3 dias.

2ª FASE (PENA INTERMEDIÁRIA)
- 0 agravante(s), 1 atenuante(s) (confissao_espontanea) [arts. 61 a 67 do CP; CP.art157]: 4 anos, 9 meses, 3 dias -> 4 anos
Pena intermediária: 4 anos.

3ª FASE (PENA DEFINITIVA)
- concurso_de_pessoas: aumento de 1/3 (em cascata) [art. 68 do CP; CP.art157.§2.II]: 4 anos -> 5 anos, 4 meses, 1 dia
- arma_de_fogo: aumento de 2/3 (em cascata) [art. 68 do CP; CP.art157.§2-A.I]: 5 anos, 4 meses, 1 dia -> 8 anos, 10 meses, 24 dias
Pena definitiva: 8 anos, 10 meses, 24 dias.

OPÇÃO DO ART. 68, PARÁGRAFO ÚNICO, DO CP
(art. 68, pará

## Casos de dosimetria (dados/casos/dosimetrias.json)

In [11]:
import json

from dosimetria import entrada_de_dict

PASTA_CASOS = next(
    p / "dados" / "casos" for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "casos").is_dir()
)


def rodar_caso(caso):
    """Converte a entrada do caso (mesmo formato da API) e roda o motor."""
    return entrada_de_dict(caso["entrada"]).calcular()


casos = json.loads((PASTA_CASOS / "dosimetrias.json").read_text(encoding="utf-8"))["casos"]
assert len(casos) >= 10, "o plano pede ao menos 10 dosimetrias no conjunto de teste"

falhas = []
for caso in casos:
    esperado = caso["esperado"]
    resultado = rodar_caso(caso)
    obtido = {
        "pena_base_dias": resultado.pena_base.dias,
        "pena_intermediaria_dias": resultado.pena_intermediaria.dias,
        "pena_definitiva_dias": resultado.pena_definitiva.dias,
        "alternativa_art68_dias": resultado.alternativa_art68 and resultado.alternativa_art68.pena_definitiva.dias,
    }
    for chave, valor in obtido.items():
        if valor != esperado[chave]:
            falhas.append(f"{caso['id']}: {chave} esperado {esperado[chave]}, obtido {valor}")
    for trecho in esperado["alertas_contem"]:
        if not any(trecho in alerta for alerta in resultado.alertas):
            falhas.append(f"{caso['id']}: nenhum alerta contém {trecho!r} (alertas: {resultado.alertas})")
    if not esperado["alertas_contem"] and resultado.alertas:
        falhas.append(f"{caso['id']}: alertas inesperados {resultado.alertas}")
    # o passo a passo sempre sai do mínimo da faixa e termina na pena definitiva
    assert resultado.passos[0].valor_antes == resultado.faixa_aplicada.minimo
    assert resultado.passos[-1].valor_depois == resultado.pena_definitiva
    print(f"{caso['id']:<16} {str(resultado.pena_definitiva):<28} {caso['descricao'][:60]}")

assert not falhas, "\n".join(falhas)

# o índice do conjunto de treinamento aponta para casos que existem no arquivo de dosimetrias
treinamento = json.loads((PASTA_CASOS / "conjunto_treinamento.json").read_text(encoding="utf-8"))["casos"]
ids = {caso["id"] for caso in casos}
assert len(treinamento) == 10
for sentenca in treinamento:
    assert sentenca["tem_dosimetria"] == (sentenca["caso_dosimetria"] is not None)
    assert sentenca["caso_dosimetria"] is None or sentenca["caso_dosimetria"] in ids

print(f"Casos de dosimetria ({len(casos)}): OK")

treinamento-04   2 anos                       Furto qualificado por concurso de pessoas; pena-base no míni
construido-01    2 anos, 3 meses, 29 dias     Furto simples em repouso noturno, réu reincidente, culpabili
construido-02    8 anos                       Roubo com concurso de pessoas e arma de fogo, consequências 
construido-03    4 anos, 9 meses, 3 dias      Homicídio simples tentado; reincidência e confissão igualmen
construido-04    1 ano, 8 meses, 3 dias       Tráfico privilegiado: confissão sem efeito pela Súmula 231, 
construido-05    1 ano, 11 meses, 31 dias     Estelionato contra entidade de direito público, três circuns
construido-06    1 ano, 9 meses, 2 dias       Receptação: reincidência preponderante contra atenuante inom
construido-07    4 anos, 2 meses              Lesão corporal grave: 8 circunstâncias desfavoráveis, agrava
construido-08    2 anos, 4 meses, 1 dia       Furto qualificado em concurso formal (art. 70, Parte Geral),
construido-09    6 anos              

## Sentenças do conjunto de treinamento (lidas do PDF)

In [12]:
from dataclasses import replace

from sentencas import eh_penal, extrair_dosimetria_declarada, ler_sentencas

RAIZ = PASTA_CASOS.parent.parent
sentencas = ler_sentencas(RAIZ / "Conjunto de Treinamento - 10 Sentenças Judiciais.pdf")
anotacao = json.loads((PASTA_CASOS / "conjunto_treinamento.json").read_text(encoding="utf-8"))["casos"]
casos_por_id = {caso["id"]: caso for caso in casos}

# o leitor separa as 10 sentenças do PDF, e cada uma bate com a anotação feita à mão
assert [s.numero for s in sentencas] == list(range(1, 11))
for sentenca, esperado in zip(sentencas, anotacao):
    assert sentenca.numero == esperado["numero"]
    assert (sentenca.ramo, sentenca.tema, sentenca.processo, sentenca.resultado) == (
        esperado["ramo"], esperado["tema"], esperado["processo"], esperado["resultado"]
    ), f"caso {sentenca.numero}"
    assert sentenca.relatorio.startswith("Vistos.")
    assert sentenca.partes and sentenca.magistrado and sentenca.cargo
    assert eh_penal(sentenca) == esperado["tem_dosimetria"]

    declarada = extrair_dosimetria_declarada(sentenca)
    if not esperado["tem_dosimetria"]:
        # as 9 sentenças não penais: nenhuma dosimetria a extrair nem a calcular
        assert declarada is None, f"caso {sentenca.numero}"
        print(f"caso {sentenca.numero:>2}: {sentenca.ramo:<26} sem dosimetria (não penal)")
        continue

    # a dosimetria que o juiz escreveu, lida do PDF
    anotada = esperado["dosimetria_declarada"]
    assert declarada.pena_base == Pena(anotada["pena_base_dias"])
    assert declarada.pena_definitiva == Pena(anotada["pena_definitiva_dias"])
    assert declarada.dias_multa == anotada["dias_multa"]
    assert declarada.regime_inicial == anotada["regime_inicial"]

    # o motor, com os fatos do caso, chega à mesma pena que a sentença
    resultado = rodar_caso(casos_por_id[esperado["caso_dosimetria"]])
    assert resultado.pena_base == declarada.pena_base, (resultado.pena_base, declarada.pena_base)
    assert resultado.pena_definitiva == declarada.pena_definitiva, (resultado.pena_definitiva, declarada.pena_definitiva)
    print(
        f"caso {sentenca.numero:>2}: {sentenca.ramo:<26} sentença: {declarada.pena_definitiva}"
        f" | motor: {resultado.pena_definitiva}  -> iguais"
    )
    print()
    print("  Trecho da sentença:", declarada.trecho[: declarada.trecho.index("Substituo")].strip())
    print()
    print("  " + gerar_fundamentacao(resultado).replace("\n", "\n  "))
    print()

# o extrator de pena também entende anos, meses e dias por extenso (formato usual de sentença)
sentenca_penal = next(s for s in sentencas if eh_penal(s))
variante = replace(
    sentenca_penal,
    dispositivo=(
        "Dosimetria: Fixo a pena-base em 4 (quatro) anos e 6 (seis) meses de reclusão. "
        "Pena definitiva em 5 (cinco) anos, 3 (três) meses e 10 (dez) dias de reclusão "
        "e 12 (doze) dias-multa. Regime inicial Semiaberto."
    ),
)
declarada = extrair_dosimetria_declarada(variante)
assert declarada.pena_base == Pena.de_anos_meses_dias(anos=4, meses=6)
assert declarada.pena_definitiva == Pena.de_anos_meses_dias(anos=5, meses=3, dias=10)
assert declarada.dias_multa == 12
assert declarada.regime_inicial == "semiaberto"

print("Sentenças do conjunto de treinamento: OK")

caso  1: Direito do Consumidor      sem dosimetria (não penal)
caso  2: Direito de Família         sem dosimetria (não penal)
caso  3: Direito do Trabalho        sem dosimetria (não penal)
caso  4: Direito Penal              sentença: 2 anos | motor: 2 anos  -> iguais

  Trecho da sentença: Dosimetria: Fixada a pena-base no mínimo legal em 2 anos de reclusão. Ausentes agravantes ou atenuantes, bem como causas de aumento/diminuição. Pena definitiva em 2 (dois) anos de reclusão e 10 dias-multa.

  DOSIMETRIA DA PENA
  
  Faixa aplicada (CP.art155.§4.IV): de 2 anos a 8 anos.
  Critério de quantum nas fases 1 e 2: fração do intervalo (1/8).
  Composição da 3ª fase: causas compostas em cascata (cada fração sobre a pena já modificada).
  
  1ª FASE (PENA-BASE)
  - nenhuma circunstância desfavorável: pena-base fixada no mínimo da faixa [art. 59 do CP; CP.art155.§4.IV]: 2 anos -> 2 anos
  Pena-base: 2 anos.
  
  2ª FASE (PENA INTERMEDIÁRIA)
  - 0 agravante(s), 0 atenuante(s) [arts. 61 a 67 do 

## Entrada e saída em JSON (formato da API)

In [13]:
from copy import deepcopy

from dosimetria import resultado_para_dict

entrada_exemplo = casos_por_id["construido-02"]["entrada"]

# o JSON de saída traz as penas em dias e em texto, os passos, os alertas e a fundamentação
saida = resultado_para_dict(entrada_de_dict(entrada_exemplo).calcular())
assert saida["pena_definitiva"] == {"total_dias": 2920, "anos": 8, "meses": 0, "dias": 0, "texto": "8 anos"}
assert saida["alternativa_art68"]["pena_definitiva"]["total_dias"] == 2433
assert saida["composicao"] == "sobre_pena_intermediaria"
assert saida["fundamentacao"].startswith("DOSIMETRIA DA PENA")
assert [p["fase"] for p in saida["passos"]][:2] == ["1ª fase (pena-base)", "2ª fase (pena intermediária)"]
json.dumps(saida)  # serializável sem conversão extra


def erro_de(alterar):
    """Aplica uma alteração numa cópia da entrada de exemplo e devolve a mensagem de erro."""
    entrada = deepcopy(entrada_exemplo)
    alterar(entrada)
    try:
        entrada_de_dict(entrada)
    except ValueError as erro:
        return str(erro)
    raise AssertionError("deveria ter rejeitado a entrada")


# erros de formato viram mensagens que dizem o campo e as opções válidas
assert "circunstancias_desfavoraveis inválido: 'culpa'" in erro_de(
    lambda e: e.update(circunstancias_desfavoraveis=["culpa"])
)
assert "opções: culpabilidade, antecedentes" in erro_de(lambda e: e.update(circunstancias_desfavoraveis=["culpa"]))
assert "composicao inválido" in erro_de(lambda e: e.update(composicao="soma"))
assert "estrategia.tipo inválido" in erro_de(lambda e: e["estrategia"].update(tipo="metade"))
assert "fração inválida: '1,3'" in erro_de(lambda e: e["causas"][0].update(fracao_min="1,3"))
assert "campo obrigatório ausente em entrada: 'faixa'" in erro_de(lambda e: e.pop("faixa"))
assert "campos desconhecidos" in erro_de(lambda e: e["faixa"].update(minimo={"semanas": 3}))
assert "valores negativos" in erro_de(lambda e: e["faixa"].update(minimo={"anos": -1}))
# regras do motor continuam valendo: fração acima da mínima sem justificativa
assert "exige justificativa" in erro_de(lambda e: e["causas"][0].update(fracao_escolhida="1/2"))

print("Entrada e saída em JSON: OK")

Entrada e saída em JSON: OK


## API (FastAPI)

In [14]:
from fastapi.testclient import TestClient

from api.app import app
from api.esquemas import EXEMPLO_ENTRADA, PedidoComparacao

cliente = TestClient(app)

# rotas gerais e documentação interativa
assert cliente.get("/saude").json() == {"status": "ok"}
assert cliente.get("/docs").status_code == 200
assert cliente.get("/openapi.json").json()["info"]["title"] == "sergius-ia-Judge"
assert cliente.get("/opcoes").json()["composicoes"] == ["cascata", "sobre_pena_intermediaria"]

# os exemplos da API são os mesmos casos testados acima, e cada um dá o resultado esperado
exemplos = cliente.get("/exemplos").json()
assert [e["id"] for e in exemplos] == [c["id"] for c in casos]
for caso in casos:
    entrada = cliente.get(f"/exemplos/{caso['id']}").json()["entrada"]
    resposta = cliente.post("/dosimetria/calcular", json=entrada)
    assert resposta.status_code == 200, (caso["id"], resposta.json())
    corpo = resposta.json()
    assert corpo["pena_definitiva"]["total_dias"] == caso["esperado"]["pena_definitiva_dias"], caso["id"]
    alternativa = corpo["alternativa_art68"]
    assert (alternativa and alternativa["pena_definitiva"]["total_dias"]) == caso["esperado"]["alternativa_art68_dias"]
assert cliente.get("/exemplos/nao-existe").status_code == 404

# o exemplo mostrado na documentação funciona
corpo = cliente.post("/dosimetria/calcular", json=EXEMPLO_ENTRADA).json()
assert corpo["pena_definitiva"]["texto"] == "2 anos, 3 meses, 29 dias"
assert corpo["fundamentacao"].startswith("DOSIMETRIA DA PENA")

# correção do estudante: acerta as fases 1 e 2, erra a 3ª por 1 dia
pedido = PedidoComparacao.model_config["json_schema_extra"]["examples"][0]
correcao = cliente.post("/ensino/comparar", json=pedido).json()
assert (correcao["acertos"], correcao["total"]) == (2, 3)
assert [f["correta"] for f in correcao["fases"]] == [True, True, False]
assert correcao["fases"][2]["diferenca_dias"] == 1
assert correcao["fases"][2]["explicacao"] == ["repouso_noturno: aumento de 1/3 (em cascata)"]

# na pena definitiva, a opção do art. 68, parágrafo único, também conta como certa
roubo = cliente.get("/exemplos/construido-02").json()["entrada"]
opcao_limitada = {"pena_definitiva": {"anos": 6, "meses": 8, "dias": 3}}  # 2433 dias
correcao = cliente.post("/ensino/comparar", json={"entrada": roubo, "resposta": opcao_limitada}).json()
assert correcao["fases"][0]["correta"] and "art. 68, parágrafo único" in correcao["fases"][0]["observacao"]
assert cliente.post("/ensino/comparar", json={"entrada": roubo, "resposta": {}}).status_code == 422

# erros de formato: 422 com o campo e a mensagem em português
entrada_ruim = json.loads(json.dumps(roubo))
entrada_ruim["composicao"] = "soma"
entrada_ruim["causas"][0]["fracao_min"] = "1,3"
del entrada_ruim["estrategia"]
erros = {e["campo"]: e["mensagem"] for e in cliente.post("/dosimetria/calcular", json=entrada_ruim).json()["detail"]}
assert erros == {
    "composicao": "valor inválido (opções: 'cascata', 'sobre_pena_intermediaria')",
    "causas.0.fracao_min": "formato inválido (frações no formato '1/3')",
    "estrategia": "campo obrigatório ausente",
}

# regras do motor também voltam como 422 com a explicação
sem_justificativa = json.loads(json.dumps(roubo))
sem_justificativa["causas"][0]["fracao_escolhida"] = "1/2"
resposta = cliente.post("/dosimetria/calcular", json=sem_justificativa)
assert resposta.status_code == 422
assert resposta.json()["detail"] == "concurso_de_pessoas: fração 1/2 acima da mínima exige justificativa"

print("API: OK")

API: OK
